# Exclude large events from every downstream analysis

Events exceeding MAX_CELLS grid cells are removed, and filtered copies of the files are saved. Events larger than 90 grid cells were frequently associated with major urban areas, suggesting that these events may reflect urban NO2 enhancements

In [1]:
import os, pandas as pd
from pathlib import Path

/global/home/users/rubypercival/.local/lib/python3.11/site-packages/pandas/core/computation/expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/global/home/users/rubypercival/.local/lib/python3.11/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


In [2]:
YEAR = 2024
MAX_CELLS = 90 # exclude events with more than this many grid cells
EXTRA_IDS = []  # other group_ids to exclude 

SRC = Path("/global/scratch/users/rubypercival/0.1x0.1_deg_gridded_datasets/final_datasets")
DST = Path("/global/scratch/users/rubypercival/0.1x0.1_deg_gridded_datasets/final_datasets_excl")
DST.mkdir(parents=True, exist_ok=True)
files = {"grouped": f"lowCF_grouped_2hour_pulses_MCD12Q1mask_{YEAR}_post10am.parquet",
         "merged": f"lowCF_filtered_merged_with_incdec_MCD12Q1mask_{YEAR}.parquet",
         "events_long": f"lowCF_7day_events_long_MCD12Q1mask_{YEAR}.parquet"}

def dst(name):               
    p = DST / name
    if p.is_symlink(): p.unlink()
    return p

key = lambda d, t: list(zip(d["grid_lat"].astype(float).round(2), d["grid_lon"].astype(float).round(2), t))
is_true = lambda s: s.astype(str).eq("True")

In [3]:
g = pd.read_parquet(SRC / files["grouped"])
pul = g[is_true(g["pulse_start"])]
size = pul.drop_duplicates(["group_id", "grid_lat", "grid_lon"]).groupby("group_id").size()
ids = set(size[size > MAX_CELLS].index) | set(EXTRA_IDS)
missing = set(EXTRA_IDS) - set(size.index)
assert not missing, f"{YEAR}: EXTRA_IDS not in the grouped file: {sorted(missing)}"

drop = pul[pul["group_id"].isin(ids)].copy()
pulse_keys = set(key(drop, pd.to_datetime(drop["datetime"]).dt.floor("h")))   # cell + UTC hour of each excluded pulse
day_keys = set(key(drop, pd.to_datetime(drop["datetime_local"]).dt.normalize())) # cell + local day of each excluded pulse

log = (drop.groupby("group_id").agg(start_local=("datetime_local", "min"), lat=("grid_lat", "mean"), lon=("grid_lon", "mean"),
                                    pulse_starts=("grid_lat", "size"))
       .join(size.rename("cells")).assign(reason=lambda d: [f"> {MAX_CELLS} cells" if c > MAX_CELLS else "manual" for c in d["cells"]]))
print(f"{YEAR}: excluding {len(ids)} events ({len(drop):,} pulse starts)")
print(log.sort_values("cells", ascending=False).round({"lat": 2, "lon": 2}).to_string())

2024: excluding 5 events (632 pulse starts)
                 start_local    lat    lon  pulse_starts  cells      reason
group_id                                                                   
1709     2024-04-09 10:00:00  47.85 -96.52           161    136  > 90 cells
3527     2024-05-30 12:00:00  41.43 -88.99           135    126  > 90 cells
1156     2024-03-27 12:00:00  40.77 -97.80           131    123  > 90 cells
2557     2024-04-24 11:00:00  42.88 -89.27           110     98  > 90 cells
2365     2024-04-24 12:00:00  40.88 -88.44            95     92  > 90 cells


In [4]:
# grouped file
g_out = g[~g["group_id"].isin(ids)]
g_out.to_parquet(dst(files["grouped"]), index=False)

# merged file
m = pd.read_parquet(SRC / files["merged"])
hit = pd.Series(key(m, pd.to_datetime(m["datetime"]).dt.floor("h")), index=m.index).isin(pulse_keys) & is_true(m["pulse_start"])
n_before = int(is_true(m["pulse_start"]).sum())
m.loc[hit, "pulse_start"] = False if m["pulse_start"].dtype == bool else "False"
n_after = int(is_true(m["pulse_start"]).sum())
m.to_parquet(dst(files["merged"]), index=False)
del m

# events_long
el = pd.read_parquet(SRC / files["events_long"])
d0 = el[el["lag_days"] == 0].drop_duplicates("event_id")
bad = d0.loc[pd.Series(key(d0, pd.to_datetime(d0["datetime_local"]).dt.normalize()), index=d0.index).isin(day_keys), "event_id"]
el_out = el[~el["event_id"].isin(bad)]
el_out.to_parquet(dst(files["events_long"]), index=False)

for p in SRC.iterdir():
    if not (DST / p.name).exists() and not (DST / p.name).is_symlink():
        os.symlink(p, DST / p.name)
log.to_csv(DST / f"excluded_events_{YEAR}.csv")

print(f"{YEAR}: grouped {g['group_id'].nunique():,} → {g_out['group_id'].nunique():,} events, "
      f"{int(is_true(g['pulse_start']).sum()):,} → {int(is_true(g_out['pulse_start']).sum()):,} pulse starts")
print(f"merged pulse_start {n_before:,} → {n_after:,} (un-flagged {int(hit.sum()):,} of {len(pulse_keys):,})")
print(f"events_long windows {el['event_id'].nunique():,} → {el_out['event_id'].nunique():,}")

2024: grouped 1,307 → 1,302 events, 4,396 → 3,764 pulse starts
merged pulse_start 16,583 → 15,951 (un-flagged 632 of 632)
events_long windows 16,583 → 15,887
